#Problem Statement

#Document Classification on NewsGroup Dataset


Completion requirements
This Lab tutorial explains Document Classification through TF*IDF on NewGroup Dataset,
Run this code and note the accuracy.

For the portfolio, looking at Lab1 code, modify Lab 2 code to implement Word2Vector model and compare the accuracy



In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("CN7030_ML_ON_BIGDATA") \
    .config("spark.some.config.option", "some-value") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

In [3]:
from sklearn.datasets import fetch_20newsgroups
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

In [4]:
newsgroups = fetch_20newsgroups(subset='all')


In [5]:
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

In [6]:
print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Categories: {newsgroups.target_names}")
print(f"Number of categories: {len(newsgroups.target_names)}")

Total number of documents: 18846
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
Number of categories: 20


In [7]:
category_counts = df.groupBy('category').count().toPandas()
print("Category distribution before filtering (25%):")
print(category_counts)


Category distribution before filtering (25%):
    category  count
0         19    628
1          0    799
2          7    990
3          6    975
4          9    994
5         17    940
6          5    988
7          1    973
8         10    999
9          3    982
10        12    984
11         8    996
12        11    991
13         2    985
14         4    963
15        13    990
16        18    775
17        14    987
18        15    997
19        16    910


In [8]:
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)

In [9]:
total_documents_after_sampling = df_sampled.count()
print(f"Total number of documents after sampling: {total_documents_after_sampling}")

Total number of documents after sampling: 4773


In [10]:
# Prepare for Document Classification

# Step 1: Tokenize the text
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# Step 2: Apply HashingTF
hashingTF = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=1000)

# Step 3: Compute IDF (Inverse Document Frequency)
idf = IDF(inputCol="raw_features", outputCol="features")

# Step 4: Convert category labels to numerical labels
indexer = StringIndexer(inputCol="category", outputCol="label")

# Step 5: Define the classifier (Logistic Regression in this case)
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Set up the pipeline with all the stages
pipeline = Pipeline(stages=[tokenizer, hashingTF, idf, indexer, lr])

# Split the data into training and testing sets (80% train, 20% test)
train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)

# Step 6: Train the model using the pipeline
model = pipeline.fit(train_data)

# Step 7: Make predictions on the test data
predictions = model.transform(test_data)

# Show some of the predictions
predictions.select("text", "category", "prediction").show(5, truncate=False)

# Step 8: Evaluate the model's accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

# Display the accuracy
print(f"Model Accuracy: {accuracy:.2f}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
import numpy as np
# Apply the pipeline to the sampled data (df_sampled) to get the 'features' column (TF-IDF vectors)
processed_data = model.transform(df_sampled)

# Extract the "features" column as an RDD
tdm_rdd = processed_data.select("features").rdd.map(lambda row: row[0])

# Convert the RDD of vectors into a numpy array
tdm_array = np.array(tdm_rdd.collect())

# Convert the numpy array into a DataFrame (this is our Term-Document Matrix)
tdm_df = pd.DataFrame(tdm_array)

# Show the Term-Document Matrix
print("Term-Document Matrix (TDM):")
print(tdm_df)

# Optional: Display the first few rows of the TDM
print(tdm_df.head())


Term-Document Matrix (TDM):
           0         1        2    3         4    5        6         7    \
0     0.000000  0.000000  1.90774  0.0  0.000000  0.0  0.00000  0.000000   
1     0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
2     0.000000  0.000000  0.00000  0.0  2.223593  0.0  2.70702  0.000000   
3     1.983452  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4     0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
...        ...       ...      ...  ...       ...  ...      ...       ...   
4768  0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4769  1.983452  0.000000  0.00000  0.0  2.223593  0.0  0.00000  0.000000   
4770  0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4771  0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4772  0.000000  2.504946  0.00000  0.0  0.000000  0.0  0.00000  1.840601   

           8         9    ...       990  991       992  993

#Document Classification through TF*IDF on NewGroup Dataset is 55%

In [12]:
!pip install pyspark nltk

In [13]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [14]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [15]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stem_words(words):
    return [stemmer.stem(word) for word in words]

def lemmatize_words(words):
    return [lemmatizer.lemmatize(word) for word in words]

stem_udf = udf(stem_words, ArrayType(StringType()))
lemma_udf = udf(lemmatize_words, ArrayType(StringType()))

In [16]:
from pyspark.ml.feature import StopWordsRemover
from pyspark.sql.functions import col


In [17]:
data.head()

,text,category
0,From: Mamatha Devineni Ratnam <mr47+@andrew.cm...,10
1,From: mblawson@midway.ecn.uoknor.edu (Matthew ...,3
2,From: hilmi-er@dsv.su.se (Hilmi Eren)\nSubject...,17
3,From: guyd@austin.ibm.com (Guy Dawson)\nSubjec...,3
4,From: Alexander Samuel McDiarmid <am2o+@andrew...,4


In [18]:
df = spark.createDataFrame(data, ["text", "category"])

# Tokenization
tokenizer = Tokenizer(inputCol="text", outputCol="words")
df = tokenizer.transform(df)

# Stopword Removal
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
df = remover.transform(df)

# Apply Stemming
df = df.withColumn("stemmed_words", stem_udf(col("filtered_words")))

# Apply Lemmatization
df = df.withColumn("lemmatized_words", lemma_udf(col("filtered_words")))

# Show Results
df.select("text", "filtered_words", "stemmed_words", "lemmatized_words").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [19]:
hashingTF = HashingTF(inputCol="lemmatized_words", outputCol="raw_features", numFeatures=500)
df = hashingTF.transform(df)

# Compute IDF
idf = IDF(inputCol="raw_features", outputCol="features")
idf_model = idf.fit(df)
df = idf_model.transform(df)

# Show TF-IDF Features
df.select("text", "features").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
indexer = StringIndexer(inputCol="category", outputCol="label")
df = indexer.fit(df).transform(df)
df.select("category", "label").distinct().show()

+--------+-----+
|category|label|
+--------+-----+
|       2|  9.0|
|      15|  1.0|
|      19| 19.0|
|       3| 11.0|
|       8|  2.0|
|       0| 17.0|
|       6| 12.0|
|       7|  6.0|
|      16| 16.0|
|      14|  8.0|
|      13|  5.0|
|      17| 15.0|
|      11|  4.0|
|       1| 13.0|
|       4| 14.0|
|       5|  7.0|
|      18| 18.0|
|      12| 10.0|
|      10|  0.0|
|       9|  3.0|
+--------+-----+



In [22]:
from pyspark.ml.feature import Word2Vec

word2Vec = Word2Vec(vectorSize=100, minCount=1, inputCol="lemmatized_words", outputCol="featuresW2Vector")
word2Vec_model = word2Vec.fit(df)
df = word2Vec_model.transform(df)

df.select("text", "featuresW2Vector").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [23]:
train_data_w2v, test_data_w2v = df.randomSplit([0.8, 0.2], seed=42)

lr_w2v = LogisticRegression(featuresCol="featuresW2Vector", labelCol="label")
lr_w2v_model = lr_w2v.fit(train_data_w2v)

predictions_w2v = lr_w2v_model.transform(test_data_w2v)
predictions_w2v.select("text", "category", "prediction").show(truncate=False)

accuracy_w2v = evaluator.evaluate(predictions_w2v)
print(f"Word2Vec Model Accuracy: {accuracy_w2v:.2f}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


## Accuracy Comparison

| Model                          | Feature Extraction          | Accuracy       |
| ------------------------------ | --------------------------- | -------------- |
| Logistic Regression + TF-IDF   | Tokenizer + HashingTF + IDF | **0.55 (55%)** |
| Logistic Regression + Word2Vec | Word2Vec Embeddings         | **0.72 (72%)** |

### Performance Comparison

* **TF-IDF Model:** **55% Accuracy**
* **Word2Vec Model:** **72% Accuracy**
* **Improvement:** **17 percentage points**

The Word2Vec-based model clearly outperformed the TF-IDF model.




## Objective

The objective of this experiment was to perform document classification using PySpark by comparing two different text feature extraction techniques: **TF-IDF** and **Word2Vec**. Logistic Regression was used as the classifier for both approaches.

## Data Preprocessing

The text data was preprocessed using the following steps:

* Tokenization of text documents.
* Label encoding of the document categories using `StringIndexer`.
* Splitting the dataset into 80% training data and 20% testing data.

## Models Implemented

### Model 1: TF-IDF + Logistic Regression

This model used:

* Tokenizer
* HashingTF
* IDF (Inverse Document Frequency)
* Logistic Regression

TF-IDF converts documents into sparse vectors based on the importance of words within the corpus.

**Accuracy:** **55%**

### Model 2: Word2Vec + Logistic Regression

This model used:

* Tokenizer
* Word2Vec
* Logistic Regression

Word2Vec generates dense vector representations that capture semantic relationships between words, allowing the classifier to better understand document meaning.

**Accuracy:** **72%**

## Comparison

| Model                          | Accuracy |
| ------------------------------ | -------: |
| TF-IDF + Logistic Regression   |  **55%** |
| Word2Vec + Logistic Regression |  **72%** |

The Word2Vec model achieved an improvement of **17 percentage points** over the TF-IDF model.

## Conclusion

The experimental results show that the **Word2Vec + Logistic Regression** model performed better than the **TF-IDF + Logistic Regression** model. While TF-IDF represents documents using word frequencies, Word2Vec captures semantic similarities between words through dense embeddings. This richer representation enabled the classifier to distinguish document categories more effectively, resulting in higher classification accuracy. Therefore, **Word2Vec** is the preferred feature extraction technique for this document classification task, achieving an accuracy of **72%** compared to **55%** for the TF-IDF approach.
